# Train the page-orientation CNN on Kaggle

Select a GPU accelerator and enable Internet in Kaggle's notebook settings. Put the source in `/kaggle/working/Production-master` or set `PROJECT_DIR` to the checkout's path in an attached Kaggle input.

Set `repo_url` to a public Hugging Face dataset tree containing Parquet shards with `image_bytes` and `class` columns. Use `non_flipped` and `flipped`, or `0` and `1`, as labels. The trainer processes one shard per epoch and resumes from `weights.pth` when you reuse the output directory. Set `epochs` to the total you want to reach, including completed epochs.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Production-master")
OUTPUT_ROOT = Path("/kaggle/working/kaggle_train")

CONFIG = {
    "device": "cuda:0",
    "side_job": {
        "repo_url": "",  # https://huggingface.co/datasets/OWNER/DATASET/tree/main/train
        "channels": [64, 128, 256, 512],
        "blocks_per_stage": [2, 2, 2, 2],
        "hidden_size": 512,
        "image_size": 224,
        "batch_size": 16,
        "prefetch_batches": 2,
        "passes_per_shard": 1,
        "cache_limit_mb": 2048,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "epochs": 1,
        "use_amp": True,
        "seed": 42,
    },
}

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

from packaging.requirements import Requirement

source = PROJECT_DIR.resolve() / "src"
requirements = PROJECT_DIR / "requirements.txt"
if not (source / "workrunner" / "aux_training.py").is_file() or not requirements.is_file():
    raise FileNotFoundError(f"Set PROJECT_DIR to the Production-master checkout. Checked {PROJECT_DIR}.")

classifier_packages = {"torch", "numpy", "pillow", "pyarrow"}
to_install = []
for line in requirements.read_text(encoding="utf-8").splitlines():
    line = line.partition("#")[0].strip()
    if not line:
        continue
    requirement = Requirement(line)
    name = requirement.name.lower()
    if name not in classifier_packages:
        continue
    try:
        installed = version(requirement.name)
    except PackageNotFoundError:
        installed = None
    if installed is not None and installed in requirement.specifier:
        continue
    if name == "torch":
        raise RuntimeError(f"Select a Kaggle GPU runtime with {requirement}.")
    to_install.append(str(requirement))

if to_install:
    subprocess.run([sys.executable, "-m", "pip", "install", *to_install], check=True)

sys.dont_write_bytecode = True
if str(source) not in sys.path:
    sys.path.insert(0, str(source))

In [ ]:
from contextlib import closing

import torch
import workrunner
from workrunner.aux_training import SideJob

if Path(workrunner.__file__).resolve().parent != source / "workrunner":
    raise RuntimeError("Restart the kernel after changing PROJECT_DIR.")

side_job = CONFIG["side_job"]
if not side_job["repo_url"].strip():
    raise ValueError("Set CONFIG['side_job']['repo_url'] in the configuration cell.")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator in Kaggle's notebook settings.")

settings = {**CONFIG, "side_job": {**side_job, "vpn": {"enabled": False}}}
with closing(SideJob(settings, project_root=OUTPUT_ROOT)) as job:
    reported_epoch = job.epoch
    while job.step():
        if job.epoch != reported_epoch:
            print(job.status(), flush=True)
            reported_epoch = job.epoch
    print(job.status(), flush=True)
    print(f"Checkpoint saved to {job.directory / 'weights.pth'}")